# 04 - Introduction to MCP with Google ADK

In the previous section, we gave a Google ADK agent access to **Python functions as tools**.

That worked well for simple demos, but those tools lived directly inside the notebook.

## From local tools to external tools
When we want tools to be reusable across applications, notebooks, and assistants, we need a more standard way to expose them. This is where **MCP** comes in.

MCP stands for **Model Context Protocol**. It gives agents a standard way to discover and use external tools.

In this notebook, ADK runs the agent and connects it to an MCP server through `McpToolset`.


## Setup

Before running the below cells, ensure you have:

1. Authenticated with your WIF-backed credentials, for example through `gcloud auth application-default login` or your workshop's WIF setup
2. Set your GCP project and location below
3. Placed `server.py` next to this notebook in `app/server.py`

This notebook does **not** use API keys. ADK is configured to use Vertex AI and Application Default Credentials.


In [ ]:
import json
import os
import sys
import uuid
from dataclasses import dataclass
from pathlib import Path

from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from google.genai.types import GenerateContentConfig, Content, Part

from mcp import StdioServerParameters

Keeping the same agent structure as the previous notebook, but using an MCP toolset instead of local Python functions.


In [ ]:
# Select the model
MODEL_NAME = "gemini-2.5-flash"
APP_NAME = "Travel-Assistant"
USER_ID = "workshop-user"

# Set a default instruction for the agent
SYSTEM_MESSAGE = """
You are a helpful travel assistant.
You have access to tools. Decide whether you need to use a tool.
- If needed, use it
- Otherwise, answer directly
"""


@dataclass
class AgentResponse:
    text: str
    function_calls: list[dict]


async def ask_agent(
    prompt: str,
    tools: list | None = None,
    system_instruction: str = SYSTEM_MESSAGE,
    temperature: float = 0.7,
    top_k: int = 40,
    top_p: float = 1.0,
) -> AgentResponse:
    """Run one ADK agent turn and return the final text plus tool calls."""

    config = GenerateContentConfig(
        temperature=temperature,                # <-- Controls randomness: 0=deterministic, 1=creative, 2=very random
        top_p=top_p,                            # <-- Nucleus sampling: considers tokens with cumulative probability up to this value
        top_k=top_k,                            # <-- Limits sampling to the top K most likely tokens at each step
    )

    agent = Agent(
        name="travel_assistant",
        model=MODEL_NAME,
        instruction=system_instruction,
        tools=tools or [],
        generate_content_config=config
    )

    session_service = InMemorySessionService()
    session_id = f"session-{uuid.uuid4().hex}"
    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=session_id,
    )

    runner = Runner(
        app_name=APP_NAME,
        agent=agent,
        session_service=session_service,
    )

    message = Content(
        role="user",
        parts=[Part(text=prompt)],
    )

    final_text = ""
    function_calls = []

    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=session_id,
        new_message=message,
    ):
        for call in event.get_function_calls():
            function_calls.append({"name": call.name, "args": dict(call.args or {})})

        if event.is_final_response() and event.content and event.content.parts:
            text_parts = [part.text for part in event.content.parts if part.text]
            if text_parts:
                final_text = "\n".join(text_parts)

    return AgentResponse(text=final_text, function_calls=function_calls)


## Connect to the MCP server

The MCP server lives in `app/server.py`. ADK starts it as a local process through `McpToolset`, then exposes the server's MCP tools to the agent.


In [ ]:
SERVER_SCRIPT = Path("app/server.py").resolve()

mcp_tools = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command=sys.executable,
            args=[str(SERVER_SCRIPT)],
        ),
        timeout=10.0,
    )
)

print("MCP toolset configured.")


## Discover available tools

One benefit of MCP is that the client can ask the server which tools it provides.

This makes the tool interface discoverable and reusable.

In [ ]:
available_tools = await mcp_tools.get_tools()

for tool in available_tools:
    print(f"- {tool.name}: {tool.description}")


## Tool call through MCP

We now ask the same kind of question as before.

The difference is that the model is no longer calling Python functions from the notebook.
It is using tools exposed by the MCP server.

In [ ]:
prompt = """
I am visiting Lisbon this weekend. What flights are available from Amsterdam?
"""

response = await ask_agent(
    prompt,
    tools=[mcp_tools],
)

print("Response:")
print(response.text)


When No MCP tool is invoked as no relevant tools are available for the query

In [ ]:
prompt = """
I am visiting Lisbon this weekend. What should I pack?
"""

response = await ask_agent(
    prompt,
    tools=[mcp_tools],
)

print("Response:")
print(response.text)

## Inspect the tool calls

The final text is useful, but for learning it is even more useful to inspect which tool calls the model decided to make.

In [ ]:
print(json.dumps(response.function_calls, indent=2, ensure_ascii=False))


## Cleanup

Close the MCP toolset when you are done so the local server process is stopped.


In [ ]:
await mcp_tools.close()